# Test the Gabrielle model on another NZ event

Run cells in order in Colab. Choose **Runtime → Change runtime type → T4 GPU**.

1. Set your own Earth Engine project ID below.
2. Start with `EVENT = 'dovi'`. Ruby and Gretel are also available.
3. Your existing `outputs/unet_best.pt` is loaded without retraining.

Outputs go to a separate `external_tests/<event>/` folder on Drive. This notebook produces **flood predictions**, not independently validated flood accuracy. No IoU is calculated because observed reference masks are not yet available. Gretel is an imagery screening case; a NZ flood has not been confirmed.


In [ ]:
%pip install -q earthengine-api geemap geedim rasterio segmentation-models-pytorch


In [ ]:
from pathlib import Path
import json
import numpy as np
import torch
import segmentation_models_pytorch as smp
import rasterio
from rasterio.windows import Window
from google.colab import drive

EVENT = 'dovi'  # 'dovi', 'ruby', or 'gretel'
EE_PROJECT = ''  # REQUIRED: enter the project ID that worked in data_collection.ipynb
DRIVE_DIR = Path('/content/drive/MyDrive/Coastal-Change-Detection')
CHECKPOINT_PATH = DRIVE_DIR / 'outputs/unet_best.pt'

EVENTS = {
    'dovi': dict(area='Wairarapa', bbox=[175.0, -41.5, 176.1, -40.7],
                 pre='2022-02-01', post='2022-02-13', orbit=175, direction='DESCENDING', crs='EPSG:32760'),
    'ruby': dict(area='East Cape', bbox=[177.8, -37.85, 178.6, -37.45],
                 pre='2021-12-03', post='2021-12-15', orbit=175, direction='DESCENDING', crs='EPSG:32760'),
    'gretel': dict(area='Northland', bbox=[173.0, -35.8, 174.6, -34.4],
                   pre='2020-03-07', post='2020-03-19', orbit=81, direction='ASCENDING', crs='EPSG:32759'),
}
if EVENT not in EVENTS:
    raise ValueError('Choose dovi, ruby, or gretel.')
if not EE_PROJECT.strip():
    raise ValueError('Enter your Earth Engine project ID in EE_PROJECT, then rerun this cell.')
config = EVENTS[EVENT]
drive.mount('/content/drive')
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Model not found: {CHECKPOINT_PATH}')
OUT_DIR = DRIVE_DIR / 'external_tests' / EVENT
OUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Event:', EVENT, config['area'])
print('Acquisition dates are UTC:', config['pre'], '→', config['post'])
print('Output folder:', OUT_DIR)


In [ ]:
# Load only the trusted checkpoint generated by your training notebook.
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
CHANNELS = ['pre_vv', 'pre_vh', 'post_vv', 'post_vh']
if checkpoint['input_bands'] != CHANNELS or checkpoint['in_channels'] != 4:
    raise ValueError('Checkpoint inputs do not match the four-channel collection below.')
mean = np.asarray(checkpoint['normalisation']['mean'], dtype=np.float32)
std = np.asarray(checkpoint['normalisation']['std'], dtype=np.float32)
if mean.shape != (4,) or std.shape != (4,) or not np.isfinite(mean).all() or not np.isfinite(std).all() or (std <= 0).any():
    raise ValueError('Invalid checkpoint normalisation statistics.')
THRESHOLD = float(checkpoint['threshold'])
if not 0 <= THRESHOLD <= 1:
    raise ValueError('Invalid saved decision threshold.')
model = smp.Unet(encoder_name=checkpoint['encoder'], encoder_weights=None,
                 in_channels=4, classes=1)
model.load_state_dict(checkpoint['model_state'])
model = model.to(device).eval()
print('Loaded model from epoch', checkpoint['epoch'])
print('Channel order:', CHANNELS)
print('Saved threshold:', THRESHOLD)
print('Using original training mean/std; no retraining or refitting.')


In [ ]:
import ee
import geemap
from datetime import datetime, timedelta

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
aoi = ee.Geometry.BBox(*config['bbox'])
collection = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(aoi)
              .filter(ee.Filter.eq('instrumentMode', 'IW'))
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
              .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
              .filter(ee.Filter.eq('relativeOrbitNumber_start', config['orbit']))
              .filter(ee.Filter.eq('orbitProperties_pass', config['direction'])))
scene_ids = {}
coverages = {}
def acquisition(date, role):
    end = (datetime.fromisoformat(date) + timedelta(days=1)).date().isoformat()
    scenes = collection.filterDate(date, end)
    ids = scenes.aggregate_array('system:index').getInfo()
    if not ids:
        raise ValueError(f'No scenes for {role} on {date}. Do not continue.')
    scene_ids[role] = ids
    print(role, date, len(ids), 'slice(s):', ids)
    image = scenes.mosaic().clip(aoi)
    valid = image.select(['VV', 'VH']).mask().reduce(ee.Reducer.min()).gt(0).unmask(0, False)
    coverage = valid.reduceRegion(ee.Reducer.mean(), aoi, scale=100, maxPixels=1e9).getInfo()
    pct = 100 * next(iter(coverage.values()))
    coverages[role] = pct
    print(f'{role} AOI coverage: {pct:.1f}%')
    if pct < 95:
        raise ValueError('Less than 95% coverage. Adjust the AOI or acquisition pair before exporting.')
    return image
pre = acquisition(config['pre'], 'pre')
post = acquisition(config['post'], 'post')
# Match the training notebook: circular 30 m focal mean on dB measurements.
stack = ee.Image.cat([
    pre.select('VV').focalMean(30, 'circle', 'meters').rename('pre_vv'),
    pre.select('VH').focalMean(30, 'circle', 'meters').rename('pre_vh'),
    post.select('VV').focalMean(30, 'circle', 'meters').rename('post_vv'),
    post.select('VH').focalMean(30, 'circle', 'meters').rename('post_vh'),
]).toFloat()
print('Collection ready. These images are not independently labelled flood data.')


In [ ]:
import shutil
INPUT_PATH = OUT_DIR / f'{EVENT}_four_band_inputs.tif'
local_path = Path('/content') / f'{EVENT}_four_band_inputs.tif'
# Large areas can take several minutes. Download locally first, then copy to Drive.
geemap.download_ee_image(stack, filename=str(local_path), region=aoi,
                         scale=10, crs=config['crs'])
with rasterio.open(local_path) as src:
    if src.count != 4 or list(src.descriptions) != CHANNELS:
        raise ValueError('Unexpected export channel count or order.')
    print('Dimensions:', src.width, '×', src.height, '| grid:', src.res, '| CRS:', src.crs)
shutil.copy2(local_path, INPUT_PATH)
print('Saved inputs:', INPUT_PATH)


In [ ]:
# Predict in 256-pixel windows without loading the full scene into RAM.
# Edge/invalid input pixels are filled with training means, then excluded from outputs.
PATCH_SIZE = 256
PROB_PATH = OUT_DIR / f'{EVENT}_flood_probability.tif'
MASK_PATH = OUT_DIR / f'{EVENT}_predicted_flood.tif'
valid_count = flood_count = 0
with rasterio.open(INPUT_PATH) as src:
    profile = src.profile.copy()
    profile.update(count=1, dtype='float32', nodata=-9999, compress='deflate', BIGTIFF='IF_SAFER')
    mask_profile = profile.copy()
    mask_profile.update(dtype='uint8', nodata=255)
    total = ((src.height + 255)//256) * ((src.width + 255)//256)
    from tqdm.auto import tqdm
    with rasterio.open(PROB_PATH, 'w', **profile) as prob_dst, rasterio.open(MASK_PATH, 'w', **mask_profile) as mask_dst:
        prob_dst.set_band_description(1, 'model_flood_probability')
        mask_dst.set_band_description(1, 'predicted_flood_0_or_1_255_invalid')
        progress = tqdm(total=total, desc='Predicting')
        for y in range(0, src.height, PATCH_SIZE):
            for x in range(0, src.width, PATCH_SIZE):
                h, w = min(PATCH_SIZE, src.height-y), min(PATCH_SIZE, src.width-x)
                window = Window(x, y, w, h)
                raw = src.read(window=window)
                valid = np.isfinite(raw).all(axis=0) & (src.read_masks(window=window)>0).all(axis=0) & ~(raw == 0).all(axis=0)
                prob = np.full((h, w), -9999, dtype=np.float32)
                mask = np.full((h, w), 255, dtype=np.uint8)
                if valid.any():
                    normalized = np.zeros((4, PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
                    safe = np.where(valid[None], raw, mean[:, None, None])
                    normalized[:, :h, :w] = (safe-mean[:, None, None])/std[:, None, None]
                    with torch.inference_mode():
                        logits = model(torch.from_numpy(normalized[None]).to(device))
                        predicted = torch.sigmoid(logits)[0,0,:h,:w].cpu().numpy()
                    prob[valid] = predicted[valid]
                    mask[valid] = (predicted[valid] > THRESHOLD).astype(np.uint8)
                    valid_count += int(valid.sum())
                    flood_count += int((mask == 1).sum())
                prob_dst.write(prob, 1, window=window)
                mask_dst.write(mask, 1, window=window)
                progress.update(1)
        progress.close()
if valid_count == 0:
    raise ValueError('No valid input pixels; inspect the downloaded images.')
metadata = {'event': EVENT, 'config': config, 'scene_ids': scene_ids,
            'coverage_percent': coverages, 'checkpoint': str(CHECKPOINT_PATH),
            'checkpoint_epoch': int(checkpoint['epoch']), 'channels': CHANNELS,
            'mean': mean.tolist(), 'std': std.tolist(), 'threshold': THRESHOLD,
            'valid_pixels': valid_count, 'predicted_flood_pixels': flood_count,
            'reference_labels_available': False, 'patch_size': PATCH_SIZE,
            'stride': PATCH_SIZE, 'note': 'Nonoverlapping inference tiles; boundary seams may occur. No independent accuracy score.'}
(OUT_DIR / 'run_info.json').write_text(json.dumps(metadata, indent=2))
print('Saved probability map:', PROB_PATH)
print('Saved predicted flood mask:', MASK_PATH)
print(f'Model marks {100*flood_count/valid_count:.2f}% of valid pixels as flooded (not verified flood extent).')


In [ ]:
import matplotlib.pyplot as plt
from rasterio.enums import Resampling
with rasterio.open(INPUT_PATH) as src:
    scale = min(1, 900/max(src.height, src.width))
    shape = (max(1, round(src.height*scale)), max(1, round(src.width*scale)))
    post_vv = src.read(3, out_shape=shape, masked=True, resampling=Resampling.nearest)
with rasterio.open(PROB_PATH) as src:
    probability = src.read(1, out_shape=shape, masked=True, resampling=Resampling.nearest)
with rasterio.open(MASK_PATH) as src:
    flood = src.read(1, out_shape=shape, masked=True, resampling=Resampling.nearest)
fig, axes = plt.subplots(1,3,figsize=(15,5))
axes[0].imshow(post_vv, cmap='gray', vmin=-25, vmax=0)
axes[0].set_title('Post-event VV (dB)')
im = axes[1].imshow(probability, cmap='viridis', vmin=0, vmax=1)
axes[1].set_title('Model probability (uncalibrated)')
fig.colorbar(im, ax=axes[1], fraction=0.046)
axes[2].imshow(flood, cmap='Blues', vmin=0, vmax=1)
axes[2].set_title('Predicted flooding — unverified')
for ax in axes: ax.axis('off')
fig.suptitle(f"{EVENT.title()} · {config['area']} · post-date {config['post']} UTC")
plt.tight_layout()
plt.savefig(OUT_DIR / 'prediction_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Finished. Results are in:', OUT_DIR)


## What next?

Inspect the preview and actual valid coverage first. To try Ruby or Gretel, change `EVENT` in the configuration cell and rerun from there. Outputs for a repeated run of the same event replace that event's earlier predictions; Gabrielle inputs, patches and checkpoint are untouched.

The probability is a model score, not a calibrated confidence estimate. Nonoverlapping windows can leave tile seams. Neither dark radar pixels nor the predicted flood mask are ground truth. Independent accuracy requires time-matched observed inundation labels, permanent-water exclusions and analysed-area masks; none is supplied here.

Scene discovery and reference evidence are documented in `data/nz_event_screening/README.md`. These catalogue candidates require the coverage checks above before use.
